# POS Daily CSV

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, DoubleType, IntegerType

# 1. PATH CONFIGURATION
BRONZE_POS_CSV_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/pos_transactions_batch/"
SILVER_POS_CSV_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/silver/pos_transactions_batch/"
CHECKPOINT_SILVER_CSV = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/pos_transactions_batch/"
SILVER_POS_TABLE = "mini_project_team_a3t7.silver.pos_transactions"

# 2. READ STREAM FROM BRONZE
df_bronze_csv = spark.readStream.format("delta").load(BRONZE_POS_CSV_PATH)

# 3. SILVER TRANSFORMATIONS (Removed Watermark and Masking)
df_silver_csv = (
    df_bronze_csv
    # --- TASK: DATA TYPE ENFORCEMENT ---
    .withColumn("event_timestamp", F.to_timestamp(F.col("timestamp")))
    .withColumn("quantity",        F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price",      F.col("unit_price").cast(DoubleType()))
    .withColumn("total_amount",    F.col("total_amount").cast(DoubleType()))
    
    # --- TASK: BUSINESS RULE VALIDATION ---
    .filter((F.col("quantity") > 0) & (F.col("total_amount") > 0))
    
    # --- TASK: SCHEMA STANDARDIZATION ---
    .select(
        "transaction_id", 
        "store_id", 
        "product_id", 
        "customer_id", 
        "event_timestamp", 
        "quantity", 
        "unit_price", 
        "total_amount", 
        "payment_method", 
        "channel", 
        "_ingestion_timestamp",
        F.current_timestamp().alias("_silver_processed_at")
    )
)

# 4. WRITE STREAM TO SILVER
query_silver_csv = (
    df_silver_csv.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_SILVER_CSV)
        .trigger(availableNow=True) 
        .toTable(SILVER_POS_TABLE)
)

query_silver_csv.awaitTermination()


# Purchase Order

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, DoubleType, IntegerType, DateType

# 1. PATH CONFIGURATION
BRONZE_PO_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/purchase_orders/"
CHECKPOINT_SILVER_PO = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/purchase_orders/"
SILVER_TABLE_NAME = "mini_project_team_a3t7.silver.purchase_order"

# 2. READ STREAM
df_bronze_po = spark.readStream.format("delta").load(BRONZE_PO_PATH)

# 3. TRANSFORM & ADD METADATA
df_silver_simple = (
    df_bronze_po
    .withColumn("order_date", F.col("order_date").cast(DateType()))
    .withColumn("expected_delivery_date", F.col("expected_delivery_date").cast(DateType()))
    .withColumn("actual_delivery_date", F.col("actual_delivery_date").cast(DateType()))
    .withColumn("quantity_ordered", F.col("quantity_ordered").cast(IntegerType()))
    .withColumn("unit_cost", F.col("unit_cost").cast(DoubleType()))
    .withColumn("quality_rating", F.col("quality_rating").cast(DoubleType()))
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# 4. START STREAM USING .toTable()
query = (df_silver_simple.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_SILVER_PO)
    .trigger(availableNow=True)
    .toTable(SILVER_TABLE_NAME))

query.awaitTermination()

### Customers Table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType

# 1. PATH CONFIGURATION
BRONZE_CUSTOMERS_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/customers"
CHECKPOINT_SILVER_CUST = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/customers"

# 2. READ STREAM FROM BRONZE
df_bronze_customers = spark.readStream.format("delta").load(BRONZE_CUSTOMERS_PATH)

# 3. SILVER TRANSFORMATIONS
df_silver_customers = (
    df_bronze_customers
    # DATA TYPE ENFORCEMENT
    .withColumn("first_purchase_date", F.col("first_purchase_date").cast(DateType()))
    .withColumn("total_spend", F.col("total_spend").cast(DoubleType()))
    
    # PII MASKING
    .withColumn("zip_code", F.concat(F.substring(F.col("zip_code"), 1, 2), F.lit("***")))
    
    # BUSINESS RULE VALIDATION
    .filter(F.col("total_spend") >= 0)
    
    # SCHEMA STANDARDIZATION
    .select(
        "customer_id",
        "age_group",
        "gender",
        "zip_code",
        "loyalty_tier",
        "first_purchase_date",
        "total_spend",
        "preferred_categories",
        F.current_timestamp().alias("_silver_processed_at")
    )
)

# 4. WRITE STREAM TO SILVER TABLE
query_silver_customers = (
    df_silver_customers.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_SILVER_CUST)
        .trigger(availableNow=True)
        .toTable("mini_project_team_a3t7.silver.customers")
)

query_silver_customers.awaitTermination()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, ArrayType, LongType

# 1. PATH CONFIGURATION
BRONZE_WEATHER_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/external_weather"
SILVER_WEATHER_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/silver/external_weather"
CHECKPOINT_SILVER_WEATHER = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/external_weather"

# Target Table Name in Unity Catalog
SILVER_WEATHER_TABLE = "mini_project_team_a3t7.silver.external_weather"

# 2. DEFINE JSON SCHEMA (Matches OpenWeatherMap API structure)
weather_json_schema = StructType([
    StructField("name", StringType(), True),
    StructField("main", StructType([
        StructField("temp", DoubleType(), True),
        StructField("feels_like", DoubleType(), True),
        StructField("humidity", IntegerType(), True),
        StructField("pressure", IntegerType(), True)
    ]), True),
    StructField("weather", ArrayType(StructType([
        StructField("main", StringType(), True),
        StructField("description", StringType(), True)
    ])), True),
    StructField("wind", StructType([
        StructField("speed", DoubleType(), True)
    ]), True),
    StructField("dt", LongType(), True)
])

# 3. READ STREAM FROM BRONZE
df_bronze_weather_stream = (
    spark.readStream
    .format("delta")
    .load(BRONZE_WEATHER_PATH)
)

# 4. SILVER TRANSFORMATIONS
df_silver_transformed = (
    df_bronze_weather_stream
    .withColumn("parsed_content", F.from_json(F.col("raw_content"), weather_json_schema))
    
    # Flattening, Unit Conversion, and Column Aliasing
    .select(
        F.col("parsed_content.name").alias("city"),
        F.round(F.col("parsed_content.main.temp") - 273.15, 2).alias("temp_celsius"),
        F.round(F.col("parsed_content.main.feels_like") - 273.15, 2).alias("feels_like_celsius"),
        F.col("parsed_content.main.humidity").alias("humidity_percentage"),
        F.col("parsed_content.wind.speed").alias("wind_speed_ms"),
        F.col("parsed_content.weather")[0].getItem("main").alias("weather_main"),
        F.col("parsed_content.weather")[0].getItem("description").alias("weather_condition"),
        F.to_timestamp(F.col("parsed_content.dt")).alias("observation_time"),
        "_ingestion_timestamp" 
    )
    
    # Business Logic: Filter out invalid/extreme API results
    .filter(F.col("temp_celsius").between(-60, 60))
    
    # Add Silver-level processing metadata
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# 5. WRITE STREAM TO TABLE
query_weather = (
    df_silver_transformed.writeStream
        .format("delta")
        .outputMode("append") 
        .option("checkpointLocation", CHECKPOINT_SILVER_WEATHER)
        .trigger(availableNow=True) 
        .toTable(SILVER_WEATHER_TABLE)
)

query_weather.awaitTermination()


In [0]:
%sql
-- 1. Create a temporary table with deduplicated data
CREATE OR REPLACE TABLE mini_project_team_a3t7.silver.external_weather_new AS
WITH external_weather_new AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (
      PARTITION BY city 
      ORDER BY observation_time DESC
    ) as rank
  FROM 
    mini_project_team_a3t7.silver.external_weather
)
SELECT 
  city,
  temp_celsius,
  feels_like_celsius,
  humidity_percentage,
  wind_speed_ms,
  weather_main,
  weather_condition,
  observation_time,
  _ingestion_timestamp,
  _silver_processed_at
FROM 
  external_weather_new
WHERE 
  rank = 1;



### External Holiday


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, BooleanType

# 1. PATH CONFIGURATION
BRONZE_HOLIDAY_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/bronze/external_holiday"
SILVER_HOLIDAY_PATH = "/Volumes/mini_project_team_a3t7/default/mini_project/silver/external_holiday"
CHECKPOINT_SILVER_HOLIDAY = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/silver/external_holiday"

# Target Table Name in Unity Catalog
SILVER_HOLIDAY_TABLE = "mini_project_team_a3t7.silver.external_holiday"

# 2. DEFINE HOLIDAY JSON SCHEMA
# Note: Matching the structure of the Abstract API response
holiday_schema = ArrayType(StructType([
    StructField("name", StringType(), True),
    StructField("type", StringType(), True),
    StructField("date", StringType(), True),
    StructField("week_day", StringType(), True),
    StructField("is_national_holiday", BooleanType(), True)
]))

# 3. READ STREAM FROM BRONZE
# Streaming only picks up new files/commits added to the Bronze path
df_bronze_holiday_stream = (
    spark.readStream
    .format("delta")
    .load(BRONZE_HOLIDAY_PATH)
)

# 4. SILVER TRANSFORMATIONS
df_silver_transformed = (
    df_bronze_holiday_stream
    # Parse the JSON string into an array of objects
    .withColumn("parsed_array", F.from_json(F.col("raw_content"), holiday_schema))
    
    # Flatten the array so each holiday becomes its own row
    .withColumn("holiday_detail", F.explode_outer(F.col("parsed_array")))
    
    .select(
        F.col("holiday_detail.name").alias("holiday_name"),
        F.col("holiday_detail.type").alias("holiday_type"),
        # Standardize date format
        F.to_date(F.col("holiday_detail.date"), "MM/dd/yyyy").alias("holiday_date"),
        F.col("holiday_detail.week_day").alias("day_of_week"),
        F.col("holiday_detail.is_national_holiday").alias("is_national"),
        F.col("_batch_date").alias("check_date"),
        "_ingestion_timestamp"
    )
    
    # Deduplication for the stream (Requires watermark if state is large, 
    # but works for simple append streams)
    .dropDuplicates(["holiday_name", "holiday_date", "check_date"])
    
    # Add Silver-level audit column
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# 5. WRITE STREAM TO TABLE
# .toTable() is the cleanest way to register the stream in Unity Catalog
query_holiday = (
    df_silver_transformed.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_SILVER_HOLIDAY)
        .option("path", SILVER_HOLIDAY_PATH) # Ensures the data is stored in your Volume
        .trigger(availableNow=True) # Process new data and shut down
        .toTable(SILVER_HOLIDAY_TABLE)
)

query_holiday.awaitTermination()
print(f"Holiday Silver Table successfully updated and registered: {SILVER_HOLIDAY_TABLE}")

In [0]:
# df_silver_holiday.display()